# Notebook 05 — S&P 500 Factor Research

This notebook investigates systematic return drivers that can be constructed from the S&P 500 daily OHLCV history.

Because the project currently contains a single broad-market index rather than a cross-sectional universe of individual stocks, the factor research is **time-series based**.

The notebook studies:

- Momentum
- Trend
- Mean reversion
- Volatility
- Volume
- Price-range behavior
- Breakouts
- Risk-adjusted momentum
- Factor correlations
- Forward-return relationships
- Simple factor portfolios/signals for research only

This notebook does not claim causal relationships and does not perform production backtesting. The signals are research features that will be evaluated more rigorously in later notebooks.


## 1. Imports

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

print("Imports loaded successfully.")


## 2. Configuration and Paths

In [ ]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "data").exists():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/mnt/data/quant-trading-research"),
    ]
    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            PROJECT_ROOT = candidate
            break

MASTER_PATH = PROJECT_ROOT / "data" / "raw" / "sp500_1950_present.csv"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
REPORT_DIR = PROJECT_ROOT / "reports" / "generated"

for path in [INTERIM_DIR, FIGURE_DIR, TABLE_DIR, REPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

EXPECTED_COLUMNS = [
    "Date",
    "Open",
    "High",
    "Low",
    "Close",
    "Adj.Close",
    "Volume",
]

print(f"Master dataset: {MASTER_PATH}")


## 3. Load the Validated Master Dataset

In [ ]:
if not MASTER_PATH.exists():
    raise FileNotFoundError(
        f"Master dataset not found: {MASTER_PATH}. "
        "Run Notebook 01 and Notebook 02 first."
    )

df = pd.read_csv(MASTER_PATH, low_memory=False)

if list(df.columns) != EXPECTED_COLUMNS:
    raise ValueError(
        f"Unexpected schema. Expected {EXPECTED_COLUMNS}; "
        f"received {list(df.columns)}"
    )

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

for column in EXPECTED_COLUMNS[1:]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df = (
    df.sort_values("Date")
    .drop_duplicates("Date")
    .reset_index(drop=True)
)

if df["Date"].isna().any():
    raise ValueError("Invalid dates detected.")

if df[["Open", "High", "Low", "Close", "Adj.Close", "Volume"]].isna().any().any():
    raise ValueError("Missing OHLCV values detected.")

print(f"Rows: {len(df):,}")
print(f"Date range: {df['Date'].min().date()} → {df['Date'].max().date()}")


## 4. Base Market Features

These features provide the foundation for the factor calculations.


In [ ]:
df["return_1d"] = df["Close"].pct_change()
df["log_return_1d"] = np.log(df["Close"]).diff()

df["range"] = df["High"] - df["Low"]
df["range_pct"] = df["range"] / df["Close"]

df["intraday_return"] = df["Close"] / df["Open"] - 1
df["overnight_return"] = df["Open"] / df["Close"].shift(1) - 1

print("Base market features created.")


## 5. Momentum Factors

Momentum measures the direction and magnitude of recent price movement.

Time horizons:
- 5 trading days
- 21 trading days
- 63 trading days
- 126 trading days
- 252 trading days


In [ ]:
momentum_windows = {
    "momentum_5d": 5,
    "momentum_21d": 21,
    "momentum_63d": 63,
    "momentum_126d": 126,
    "momentum_252d": 252,
}

for name, window in momentum_windows.items():
    df[name] = df["Close"] / df["Close"].shift(window) - 1

momentum_summary = df[list(momentum_windows)].describe().T

display(momentum_summary)

momentum_summary.to_csv(
    TABLE_DIR / "sp500_momentum_factor_summary.csv"
)


## 6. Trend Factors

Trend signals compare the current price with moving averages.

These are normalized so they are comparable across the long history.


In [ ]:
for window in [20, 50, 100, 200]:
    df[f"sma_{window}"] = df["Close"].rolling(window).mean()
    df[f"price_to_sma_{window}"] = (
        df["Close"] / df[f"sma_{window}"] - 1
    )

df["sma_50_vs_sma_200"] = (
    df["sma_50"] / df["sma_200"] - 1
)

df["sma_20_slope"] = (
    df["sma_20"] / df["sma_20"].shift(20) - 1
)

df["sma_50_slope"] = (
    df["sma_50"] / df["sma_50"].shift(50) - 1
)

trend_columns = [
    "price_to_sma_20",
    "price_to_sma_50",
    "price_to_sma_100",
    "price_to_sma_200",
    "sma_50_vs_sma_200",
    "sma_20_slope",
    "sma_50_slope",
]

display(df[trend_columns].describe().T)


## 7. Volatility Factors

Volatility is measured using rolling standard deviation of daily returns.

The factors are annualized using 252 trading sessions.


In [ ]:
volatility_windows = {
    "volatility_5d": 5,
    "volatility_21d": 21,
    "volatility_63d": 63,
    "volatility_126d": 126,
    "volatility_252d": 252,
}

for name, window in volatility_windows.items():
    df[name] = (
        df["return_1d"]
        .rolling(window)
        .std()
        * np.sqrt(252)
    )

df["volatility_ratio_21_252"] = (
    df["volatility_21d"] /
    df["volatility_252d"]
)

volatility_columns = list(volatility_windows) + [
    "volatility_ratio_21_252"
]

display(df[volatility_columns].describe().T)


## 8. Range / Intraday Volatility Factors

In [ ]:
df["range_pct_5d_mean"] = df["range_pct"].rolling(5).mean()
df["range_pct_21d_mean"] = df["range_pct"].rolling(21).mean()
df["range_pct_63d_mean"] = df["range_pct"].rolling(63).mean()

df["intraday_volatility_21d"] = (
    df["intraday_return"]
    .rolling(21)
    .std()
    * np.sqrt(252)
)

df["overnight_volatility_21d"] = (
    df["overnight_return"]
    .rolling(21)
    .std()
    * np.sqrt(252)
)

range_factor_columns = [
    "range_pct_5d_mean",
    "range_pct_21d_mean",
    "range_pct_63d_mean",
    "intraday_volatility_21d",
    "overnight_volatility_21d",
]

display(df[range_factor_columns].describe().T)


## 9. Volume Factors

Volume is transformed into normalized rolling measures so that long-run changes in the absolute index volume scale do not dominate the analysis.


In [ ]:
df["volume_sma_20"] = df["Volume"].rolling(20).mean()
df["volume_sma_63"] = df["Volume"].rolling(63).mean()

df["volume_ratio_20"] = (
    df["Volume"] / df["volume_sma_20"]
)

df["volume_ratio_63"] = (
    df["Volume"] / df["volume_sma_63"]
)

df["volume_zscore_63"] = (
    (
        df["Volume"] - df["Volume"].rolling(63).mean()
    )
    /
    df["Volume"].rolling(63).std()
)

volume_factor_columns = [
    "volume_ratio_20",
    "volume_ratio_63",
    "volume_zscore_63",
]

display(df[volume_factor_columns].describe().T)


## 10. Breakout Factors

Breakout features compare today's close with historical rolling highs and lows.

The previous window is used to avoid including today's own price in the reference threshold.


In [ ]:
for window in [20, 50, 100, 252]:
    previous_high = df["High"].rolling(window).max().shift(1)
    previous_low = df["Low"].rolling(window).min().shift(1)

    df[f"breakout_high_{window}d"] = (
        df["Close"] / previous_high - 1
    )

    df[f"breakout_low_{window}d"] = (
        df["Close"] / previous_low - 1
    )

df["new_20d_high"] = (
    df["Close"] >= df["High"].rolling(20).max().shift(1)
)

df["new_20d_low"] = (
    df["Close"] <= df["Low"].rolling(20).min().shift(1)
)

breakout_columns = [
    "breakout_high_20d",
    "breakout_low_20d",
    "breakout_high_50d",
    "breakout_low_50d",
    "breakout_high_100d",
    "breakout_low_100d",
    "breakout_high_252d",
    "breakout_low_252d",
]

display(df[breakout_columns].describe().T)


## 11. Mean-Reversion Factors

Mean-reversion features measure deviations from recent averages.

The standardized distance from a moving average is especially useful for comparing historical periods.


In [ ]:
df["mean_reversion_20d"] = (
    df["Close"] / df["sma_20"] - 1
)

df["mean_reversion_50d"] = (
    df["Close"] / df["sma_50"] - 1
)

df["rolling_mean_20"] = df["Close"].rolling(20).mean()
df["rolling_std_20"] = df["Close"].rolling(20).std()

df["zscore_20"] = (
    (df["Close"] - df["rolling_mean_20"])
    / df["rolling_std_20"]
)

df["rolling_mean_63"] = df["Close"].rolling(63).mean()
df["rolling_std_63"] = df["Close"].rolling(63).std()

df["zscore_63"] = (
    (df["Close"] - df["rolling_mean_63"])
    / df["rolling_std_63"]
)

mean_reversion_columns = [
    "mean_reversion_20d",
    "mean_reversion_50d",
    "zscore_20",
    "zscore_63",
]

display(df[mean_reversion_columns].describe().T)


## 12. Risk-Adjusted Momentum

Risk-adjusted momentum divides trailing return by trailing volatility.

This is a research feature, not a portfolio performance measure.


In [ ]:
df["risk_adjusted_momentum_21d"] = (
    df["momentum_21d"] /
    df["volatility_21d"].replace(0, np.nan)
)

df["risk_adjusted_momentum_63d"] = (
    df["momentum_63d"] /
    df["volatility_63d"].replace(0, np.nan)
)

df["risk_adjusted_momentum_252d"] = (
    df["momentum_252d"] /
    df["volatility_252d"].replace(0, np.nan)
)

ram_columns = [
    "risk_adjusted_momentum_21d",
    "risk_adjusted_momentum_63d",
    "risk_adjusted_momentum_252d",
]

display(df[ram_columns].describe().T)


## 13. Trend Confirmation Signals

In [ ]:
df["trend_bullish_50_200"] = (
    df["sma_50"] > df["sma_200"]
)

df["trend_price_above_200"] = (
    df["Close"] > df["sma_200"]
)

df["trend_price_above_50"] = (
    df["Close"] > df["sma_50"]
)

df["trend_confirmation_count"] = (
    df["trend_bullish_50_200"].astype(int)
    + df["trend_price_above_200"].astype(int)
    + df["trend_price_above_50"].astype(int)
)

trend_signal_summary = (
    df[
        [
            "trend_bullish_50_200",
            "trend_price_above_200",
            "trend_price_above_50",
        ]
    ]
    .mean()
    .rename("fraction_true")
    .to_frame()
)

display(trend_signal_summary)

trend_signal_summary.to_csv(
    TABLE_DIR / "sp500_trend_signal_summary.csv"
)


## 14. Forward Return Targets for Factor Research

Forward returns are created only for **factor evaluation**.

These columns are explicitly labeled as future targets and will not be used as contemporaneous predictors.

Horizons:
- 1 day
- 5 days
- 21 days
- 63 days


In [ ]:
forward_horizons = {
    "1d": 1,
    "5d": 5,
    "21d": 21,
    "63d": 63,
}

for label, horizon in forward_horizons.items():
    df[f"forward_return_{label}"] = (
        df["Close"].shift(-horizon) / df["Close"] - 1
    )

print("Forward return targets created.")


## 15. Factor Inventory

In [ ]:
factor_columns = [
    # Momentum
    *list(momentum_windows.keys()),

    # Trend
    *trend_columns,

    # Volatility
    *volatility_columns,

    # Range
    *range_factor_columns,

    # Volume
    *volume_factor_columns,

    # Breakout
    *breakout_columns,

    # Mean reversion
    *mean_reversion_columns,

    # Risk-adjusted momentum
    *ram_columns,
]

factor_columns = list(dict.fromkeys(factor_columns))

print(f"Number of factor variables: {len(factor_columns)}")
for factor in factor_columns:
    print(f" - {factor}")


## 16. Factor Missingness and Effective Sample Size

In [ ]:
factor_quality = pd.DataFrame({
    "factor": factor_columns,
    "missing_count": [
        int(df[factor].isna().sum())
        for factor in factor_columns
    ],
})

factor_quality["valid_count"] = (
    len(df) - factor_quality["missing_count"]
)

factor_quality["valid_percent"] = (
    factor_quality["valid_count"] / len(df) * 100
)

display(factor_quality)

factor_quality.to_csv(
    TABLE_DIR / "sp500_factor_quality.csv",
    index=False
)


## 17. Factor Correlation Matrix

In [ ]:
factor_correlation = df[factor_columns].corr()

display(factor_correlation)

factor_correlation.to_csv(
    TABLE_DIR / "sp500_factor_correlation_matrix.csv"
)


## 18. Factor Correlation Heatmap

In [ ]:
fig = plt.figure(figsize=(16, 13))
plt.imshow(
    factor_correlation,
    aspect="auto",
    interpolation="nearest"
)
plt.colorbar(label="Correlation")
plt.xticks(
    range(len(factor_correlation.columns)),
    factor_correlation.columns,
    rotation=90
)
plt.yticks(
    range(len(factor_correlation.index)),
    factor_correlation.index
)
plt.title("S&P 500 Research Factor Correlation Matrix")
plt.tight_layout()

path = FIGURE_DIR / "sp500_factor_correlation_heatmap.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 19. Factor-to-Forward-Return Correlations

Pearson correlations provide a first-pass screen for linear relationships between current factor values and future returns.

These are exploratory statistics, not evidence of a tradable edge.


In [ ]:
factor_forward_rows = []

for factor in factor_columns:
    for label in forward_horizons:
        target = f"forward_return_{label}"

        valid = df[[factor, target]].dropna()

        factor_forward_rows.append({
            "factor": factor,
            "horizon": label,
            "observations": len(valid),
            "pearson_correlation": (
                valid[factor].corr(valid[target])
                if len(valid) > 2
                else np.nan
            ),
        })

factor_forward_corr = pd.DataFrame(factor_forward_rows)

display(
    factor_forward_corr.sort_values(
        "pearson_correlation",
        key=lambda s: s.abs(),
        ascending=False
    ).head(30)
)

factor_forward_corr.to_csv(
    TABLE_DIR / "sp500_factor_forward_return_correlations.csv",
    index=False
)


## 20. Rank-Based Factor Relationships

Spearman correlation is added because many financial relationships may be monotonic without being linear.


In [ ]:
spearman_rows = []

for factor in factor_columns:
    for label in forward_horizons:
        target = f"forward_return_{label}"

        valid = df[[factor, target]].dropna()

        spearman_rows.append({
            "factor": factor,
            "horizon": label,
            "observations": len(valid),
            "spearman_correlation": (
                valid[factor].corr(valid[target], method="spearman")
                if len(valid) > 2
                else np.nan
            ),
        })

factor_spearman = pd.DataFrame(spearman_rows)

display(
    factor_spearman.sort_values(
        "spearman_correlation",
        key=lambda s: s.abs(),
        ascending=False
    ).head(30)
)

factor_spearman.to_csv(
    TABLE_DIR / "sp500_factor_spearman_forward_returns.csv",
    index=False
)


## 21. Factor Quantile Analysis

For each selected factor, observations are divided into five cross-time quantiles.

Because this is a single-index time-series study, this is a **temporal conditional analysis**, not a cross-sectional factor portfolio.


In [ ]:
selected_factors = [
    "momentum_21d",
    "momentum_63d",
    "momentum_252d",
    "price_to_sma_50",
    "price_to_sma_200",
    "volatility_21d",
    "volume_ratio_20",
    "zscore_20",
    "risk_adjusted_momentum_63d",
]

quantile_results = []

for factor in selected_factors:
    factor_data = df[[factor, "forward_return_21d"]].dropna().copy()

    if len(factor_data) < 100:
        continue

    try:
        factor_data["quantile"] = pd.qcut(
            factor_data[factor],
            q=5,
            labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
            duplicates="drop",
        )
    except ValueError:
        continue

    grouped = (
        factor_data
        .groupby("quantile", observed=False)["forward_return_21d"]
        .agg(
            observations="count",
            mean="mean",
            median="median",
            std="std",
            positive_rate=lambda x: (x > 0).mean(),
        )
        .reset_index()
    )

    grouped.insert(0, "factor", factor)
    quantile_results.append(grouped)

factor_quantile_results = (
    pd.concat(quantile_results, ignore_index=True)
    if quantile_results
    else pd.DataFrame()
)

display(factor_quantile_results)

factor_quantile_results.to_csv(
    TABLE_DIR / "sp500_factor_quantile_forward_returns.csv",
    index=False
)


## 22. Momentum Research Signal

A simple long/flat research signal is constructed from 252-day momentum.

This is **not** the final backtest. It is only used to understand how the factor behaves before formal walk-forward validation and backtesting.


In [ ]:
df["momentum_signal"] = (
    df["momentum_252d"] > 0
).astype(int)

df["momentum_signal_return"] = (
    df["momentum_signal"].shift(1)
    * df["return_1d"]
)

momentum_signal_stats = pd.DataFrame({
    "metric": [
        "Signal active percentage",
        "Mean daily signal return",
        "Annualized mean signal return",
        "Annualized signal volatility",
    ],
    "value": [
        df["momentum_signal"].mean(),
        df["momentum_signal_return"].mean(),
        df["momentum_signal_return"].mean() * 252,
        df["momentum_signal_return"].std() * np.sqrt(252),
    ],
})

display(momentum_signal_stats)

momentum_signal_stats.to_csv(
    TABLE_DIR / "sp500_momentum_signal_descriptive_stats.csv",
    index=False
)


## 23. Volatility-Conditioned Returns

Analyze forward returns conditional on whether current volatility is low, normal, or high relative to its historical distribution.


In [ ]:
vol_data = df[
    ["volatility_21d", "forward_return_21d"]
].dropna().copy()

vol_data["volatility_bucket"] = pd.qcut(
    vol_data["volatility_21d"],
    q=3,
    labels=["Low Volatility", "Medium Volatility", "High Volatility"],
    duplicates="drop",
)

volatility_conditioned = (
    vol_data
    .groupby("volatility_bucket", observed=False)["forward_return_21d"]
    .agg(
        observations="count",
        mean="mean",
        median="median",
        std="std",
        positive_rate=lambda x: (x > 0).mean(),
    )
    .reset_index()
)

display(volatility_conditioned)

volatility_conditioned.to_csv(
    TABLE_DIR / "sp500_volatility_conditioned_forward_returns.csv",
    index=False
)


## 24. Momentum Factor Visualization

In [ ]:
fig = plt.figure(figsize=(14, 7))
plt.plot(
    df["Date"],
    df["momentum_252d"]
)
plt.axhline(0, linestyle="--")
plt.title("252-Day S&P 500 Momentum Factor")
plt.xlabel("Date")
plt.ylabel("Trailing 252-Day Return")
plt.grid(True, alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_momentum_252d.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 25. Trend Factor Visualization

In [ ]:
fig = plt.figure(figsize=(14, 7))
plt.plot(
    df["Date"],
    df["price_to_sma_200"]
)
plt.axhline(0, linestyle="--")
plt.title("S&P 500 Price Distance from 200-Day SMA")
plt.xlabel("Date")
plt.ylabel("Close / SMA(200) - 1")
plt.grid(True, alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_price_to_sma200_factor.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 26. Volatility Factor Visualization

In [ ]:
fig = plt.figure(figsize=(14, 7))
plt.plot(
    df["Date"],
    df["volatility_21d"],
    label="21D volatility"
)
plt.plot(
    df["Date"],
    df["volatility_252d"],
    label="252D volatility"
)
plt.title("S&P 500 Short- and Long-Horizon Volatility Factors")
plt.xlabel("Date")
plt.ylabel("Annualized Volatility")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_factor_volatility_comparison.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 27. Research Factor Dataset

The enriched factor dataset is stored under `data/interim/`.

The raw master CSV remains unchanged.


In [ ]:
factor_output = INTERIM_DIR / "sp500_factor_research.parquet"

df.to_parquet(
    factor_output,
    index=False
)

print(f"Saved: {factor_output}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")


## 28. Factor Research Summary Report

In [ ]:
top_linear = (
    factor_forward_corr
    .assign(abs_corr=lambda x: x["pearson_correlation"].abs())
    .sort_values("abs_corr", ascending=False)
    .head(20)
    .drop(columns="abs_corr")
)

top_rank = (
    factor_spearman
    .assign(abs_corr=lambda x: x["spearman_correlation"].abs())
    .sort_values("abs_corr", ascending=False)
    .head(20)
    .drop(columns="abs_corr")
)

research_report = {
    "dataset": {
        "rows": int(len(df)),
        "start": df["Date"].min().strftime("%Y-%m-%d"),
        "end": df["Date"].max().strftime("%Y-%m-%d"),
    },
    "factor_count": len(factor_columns),
    "factor_names": factor_columns,
    "top_pearson_factor_forward_relationships": top_linear.to_dict(
        orient="records"
    ),
    "top_spearman_factor_forward_relationships": top_rank.to_dict(
        orient="records"
    ),
    "important_note": (
        "Relationships are exploratory time-series statistics on a single "
        "S&P 500 index. They are not evidence of causal relationships or "
        "production-trading performance."
    ),
}

report_path = REPORT_DIR / "sp500_factor_research_report.json"

report_path.write_text(
    json.dumps(research_report, indent=2, default=str),
    encoding="utf-8"
)

print(json.dumps(research_report, indent=2, default=str))
print(f"\nSaved: {report_path}")


## 29. Final Master Dataset Integrity Check

The factor research must not modify the raw acquisition dataset.


In [ ]:
master_check = pd.read_csv(
    MASTER_PATH,
    low_memory=False
)

assert list(master_check.columns) == EXPECTED_COLUMNS
assert len(master_check) == len(df)

master_dates = pd.to_datetime(
    master_check["Date"],
    errors="coerce"
)

assert master_dates.notna().all()
assert master_dates.is_unique
assert master_dates.is_monotonic_increasing

print("Raw master dataset integrity after factor research: PASS")
print(f"Master rows: {len(master_check):,}")


# Notebook 05 Complete

Notebook 05 has completed time-series factor research on the S&P 500.

### Main factor groups

- Momentum
- Trend
- Volatility
- Range/intraday volatility
- Volume
- Breakouts
- Mean reversion
- Risk-adjusted momentum

### Main analyses

- Factor quality
- Factor correlation
- Pearson factor/forward-return relationships
- Spearman factor/forward-return relationships
- Factor quantile analysis
- Momentum signal diagnostics
- Volatility-conditioned returns

### Important methodological boundary

Because the dataset is a single S&P 500 index, these are time-series factors rather than cross-sectional stock factors. Formal out-of-sample evaluation and backtesting come later.

**Next notebook:** Notebook 06 — Regime Detection.

Run Notebook 05 from top to bottom and verify its outputs before proceeding.
